# 시드 분산 검증 — Decay-SAM

**노트북 설정** (오른쪽 패널에서 확인)

| 항목 | 값 |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** |
| Persistence | Files only |
| Input | 기존 `samlab` 코드 zip 데이터셋 **첨부 필수** |

---

## 이 노트북이 하는 일

기존 rho sweep 18개 run이 전부 `seed=0` 하나뿐이었다. 그래서

> "Muon에서 rho를 0 → 0.4로 올리니 pr70이 11.33 → 6.84로 **4.49%p** 좋아졌다"

는 주장이 **진짜 rho 효과**인지, 아니면 **그 시드가 운 좋은 골짜기에 떨어진 것**인지
구분할 방법이 없다. 이 노트북은 같은 설정을 **시드 0, 1, 2로 세 번** 돌려
그 흔들림 폭을 재고, rho 효과가 노이즈보다 큰지 판정한다.

**설계**: 3 optimizer × rho{0, 0.4} × seed{0, 1, 2} = **18 run**, 각 30 epoch.
rho 중간값(0.02~0.2)은 돌리지 않는다 — 주장하는 것이 *양 끝의 차이*이므로
양 끝의 흔들림만 알면 충분하다.

**소요 시간**: 약 **3시간** (학습 2시간 50분 + 측정 15분)

맨 위부터 순서대로 실행하면 된다. `Save & Run All` 로 걸어두고 나가도 된다.

## 0. 환경 확인

In [1]:
import torch, torchvision, sys
print("python     ", sys.version.split()[0])
print("torch      ", torch.__version__)
print("torchvision", torchvision.__version__)
print("cuda       ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu        ", torch.cuda.get_device_name(0),
          f"x{torch.cuda.device_count()}")
else:
    raise SystemExit("GPU가 꺼져 있습니다. 오른쪽 패널에서 Accelerator를 GPU로 바꾸세요.")

python      3.12.13
torch       2.10.0+cu128
torchvision 0.25.0+cu128
cuda        True
gpu         Tesla T4 x2


## 1. 코드 로드

Input 패널에 붙인 `samlab` 코드 zip을 풀고 그 안으로 이동한다.
zip이 안 붙어 있으면 여기서 멈춘다.

In [2]:
import glob, os, zipfile

DST  = '/kaggle/working/samlab'
RUNS = '/kaggle/working/runs'
DATA = '/kaggle/working/data'

candidates = glob.glob('/kaggle/input/**/*.zip', recursive=True)

if candidates:
    print('찾은 zip:', candidates)
    zipfile.ZipFile(candidates[0]).extractall(DST)
    os.chdir(DST)
else:
    # zip이 아니라 Kaggle이 자동으로 풀어버린 폴더로 올라온 경우.
    # 구조가 /kaggle/input/<dataset>/... 일 수도, /kaggle/input/datasets/<dataset>/... 처럼
    # 한 단계 더 감싸져 있을 수도 있어서 os.walk로 끝까지 뒤진다.
    hit = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'sweep.py' in files or 'sam_lab' in dirs:
            hit = root
            break
    if hit is None:
        print('/kaggle/input 전체 구조:')
        for root, dirs, files in os.walk('/kaggle/input'):
            depth = root.count(os.sep) - '/kaggle/input'.count(os.sep)
            print('  ' * depth + os.path.basename(root) + '/')
            for f in files[:5]:
                print('  ' * (depth + 1) + f)
    assert hit, "Input 패널에 samlab 코드가 없습니다. 위 구조를 보고 알려주세요."
    print('발견:', hit)
    os.chdir(hit)

os.makedirs(RUNS, exist_ok=True)
print(sorted(os.listdir('.')))
for f in ('sweep.py', 'train.py', 'analyze.py', 'sam_lab/analysis.py'):
    assert os.path.exists(f), f'{f} 없음 -> 데이터셋이 옛 버전입니다'
print('\nOK -- 코드 로드 완료, 작업 폴더:', os.getcwd())

발견: /kaggle/input/datasets/shindoooonggy/samlab-s0c
['README.md', 'analyze.py', 'final_report_cell.py', 'requirements.txt', 'sam_lab', 'sweep.py', 'tests', 'train.py']

OK -- 코드 로드 완료, 작업 폴더: /kaggle/input/datasets/shindoooonggy/samlab-s0c


## 2. 실행 계획 확인

실제로 학습을 시작하기 전에 **어떤 설정 18개가 돌 것인지** 먼저 출력해 확인한다.

**LR 주의**: `sweep.py`의 `KNOWN_BEST_LR`에는 `adamw=0.01`로 적혀 있으나,
기존 rho sweep은 `best_lr.json`을 따라 실제로는 **`adamw=0.003`** 으로 돌았다
(`config.json`에서 확인). 비교가 성립하려면 기존 run과 **똑같은 LR**을 써야 하므로
아래에 실제 쓰인 값을 직접 박아 넣는다.

In [3]:
from sweep import _cfg, FINAL_EPOCHS

LR_USED = {'adamw': 0.003, 'lion': 0.0003, 'muon': 0.02}   # 기존 run이 실제로 쓴 값
SEEDS   = [0, 1, 2]
RHOS    = [0.0, 0.40]                                      # 0.0 = SAM off

overrides = dict(out_dir=RUNS, data_root=DATA, num_workers=2, max_seconds=0.0)

plan = []
for seed in SEEDS:
    for opt, lr in LR_USED.items():
        for rho in RHOS:
            kw = dict(epochs=FINAL_EPOCHS, exp='seedvar', seed=seed, **overrides)
            cfg = (_cfg(opt, lr, sam_mode='off', **kw) if rho == 0.0
                   else _cfg(opt, lr, sam_mode='decay', rho=rho, **kw))
            plan.append(cfg)

print(f'{len(plan)}개 run 예정\n')
print(f"{'optimizer':8} {'seed':>4} {'rho':>5} {'lr':>8} {'wd':>9}   run_id")
print('-' * 92)
for c in plan:
    r = 0.0 if c.sam_mode == 'off' else c.sam_rho
    print(f'{c.optimizer:8} {c.seed:4} {r:5g} {c.lr:8g} {c.weight_decay:9.4g}   {c.run_id}')

est = sum(14 if c.optimizer == 'muon' else 8 for c in plan)
print(f'\n예상 학습 시간: 약 {est//60}시간 {est%60}분')

18개 run 예정

optimizer seed   rho       lr        wd   run_id
--------------------------------------------------------------------------------------------
adamw       0     0    0.003      0.05   seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003
adamw       0   0.4    0.003      0.05   seedvar-cifar100-resnet18-adamw-sam_decay-s0-lr0.003_rho0.4
lion        0     0   0.0003    0.1667   seedvar-cifar100-resnet18-lion-sam_off-s0-lr0.0003
lion        0   0.4   0.0003    0.1667   seedvar-cifar100-resnet18-lion-sam_decay-s0-lr0.0003_rho0.4
muon        0     0     0.02      0.05   seedvar-cifar100-resnet18-muon-sam_off-s0-lr0.02
muon        0   0.4     0.02      0.05   seedvar-cifar100-resnet18-muon-sam_decay-s0-lr0.02_rho0.4
adamw       1     0    0.003      0.05   seedvar-cifar100-resnet18-adamw-sam_off-s1-lr0.003
adamw       1   0.4    0.003      0.05   seedvar-cifar100-resnet18-adamw-sam_decay-s1-lr0.003_rho0.4
lion        1     0   0.0003    0.1667   seedvar-cifar100-resnet18-lion-sam_of

## 3. 학습 — 18 run

여기가 실제 학습. `[1/18]`, `[2/18]` … 으로 진행상황이 찍힌다.
이미 끝난 run은 자동으로 건너뛰므로, 세션이 끊겨도 다시 돌리면 이어서 진행된다.

In [4]:
from train import run_one
import time

t0 = time.time()
for i, cfg in enumerate(plan, 1):
    r = 0.0 if cfg.sam_mode == 'off' else cfg.sam_rho
    el = (time.time() - t0) / 60
    print(f'\n{"="*70}')
    print(f'[{i}/{len(plan)}] {cfg.optimizer}  seed={cfg.seed}  rho={r:g}'
          f'   (경과 {el:.0f}분)')
    print("=" * 70)
    run_one(cfg)

print(f'\n\n학습 완료 — 총 {(time.time()-t0)/60:.0f}분')


[1/18] adamw  seed=0  rho=0   (경과 0분)


100%|██████████| 169M/169M [24:08<00:00, 117kB/s] 



=== seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003  [88cf1db9b9ee] ===
    device=cuda  params=11.22M (matrix 11.21M)
    optimizer=adamw lr=0.003 sam=off
    steps/epoch=195 epochs=30
[seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003] ep   1/30 train 3.778/12.1% val 4.656/13.0% sam 0.00 18s
[seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003] ep   2/30 train 3.017/24.0% val 2.857/27.9% sam 0.00 12s
[seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003] ep   3/30 train 2.403/36.5% val 2.452/37.0% sam 0.00 12s
[seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003] ep   4/30 train 2.005/44.9% val 2.122/44.2% sam 0.00 13s
[seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003] ep   5/30 train 1.734/51.4% val 1.952/48.2% sam 0.00 13s
[seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003] ep   6/30 train 1.539/56.3% val 1.825/50.9% sam 0.00 13s
[seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003] ep   7/30 train 1.379/60.3% val 1.614/55.7% sam 0.00 13s
[seedvar-cifar100-resnet18-adamw-sam_off

## 4. 측정 — 평탄도 + 강건성

18개 모델 각각에 대해 λmax / SAM gap / basin width 와
양자화·가지치기·노이즈 강건성을 측정한다. 약 15분.

In [5]:
from analyze import probe_all

probes = probe_all(RUNS, exp='seedvar')
print(f'\n측정 완료: {len(probes)}개 모델')

probe set: 8 batches (2048 images)
[probe] seedvar-cifar100-resnet18-adamw-sam_decay-s0-lr0.003_rho0.4 ... lambda_max=22.09  sam_gap=0.7041  basin=0.867
[probe] seedvar-cifar100-resnet18-adamw-sam_decay-s1-lr0.003_rho0.4 ... lambda_max=19.95  sam_gap=0.749  basin=0.867
[probe] seedvar-cifar100-resnet18-adamw-sam_decay-s2-lr0.003_rho0.4 ... lambda_max=18.4  sam_gap=0.6041  basin=0.867
[probe] seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003 ... lambda_max=70.03  sam_gap=0.08861  basin=0.667
[probe] seedvar-cifar100-resnet18-adamw-sam_off-s1-lr0.003 ... lambda_max=79.66  sam_gap=0.07708  basin=0.667
[probe] seedvar-cifar100-resnet18-adamw-sam_off-s2-lr0.003 ... lambda_max=36.78  sam_gap=0.07502  basin=0.733
[probe] seedvar-cifar100-resnet18-lion-sam_decay-s0-lr0.0003_rho0.4 ... lambda_max=39.26  sam_gap=1.356  basin=0.800
[probe] seedvar-cifar100-resnet18-lion-sam_decay-s1-lr0.0003_rho0.4 ... lambda_max=21.42  sam_gap=0.8958  basin=0.800
[probe] seedvar-cifar100-resnet18-lion-sam_decay

## 5. 판정

핵심 출력. 세 가지를 본다.

1. **시드별 pr70 표** — 같은 설정을 시드만 바꿔 돌렸을 때 얼마나 흔들리는가
2. **판정** — rho 효과가 그 흔들림보다 큰가
3. **부수 결과** — 골짜기(시드)마다 평탄도가 다른가 (SGDR 아이디어 판단용)

판정 기준:

| 비율 | 뜻 |
|---|---|
| **2배 이상** | 효과가 노이즈보다 확실히 큼 → 기존 주장 유지 가능 |
| **1~2배** | 애매 → 시드를 더 늘려야 |
| **1배 미만** | 노이즈가 더 큼 → 주장을 약하게 고쳐야 |

In [6]:
import json
import numpy as np

probes = json.load(open(f'{RUNS}/probes.json'))
probes = {k: v for k, v in probes.items() if k.startswith('seedvar-')}

def rho_of(p):
    return 0.0 if p.get('sam_mode') == 'off' else float(p.get('rho', 0.0))

pr70, lam = {}, {}
for p in probes.values():
    key = (p['optimizer'], rho_of(p))
    if 'prune70_drop' in p:
        pr70.setdefault(key, {})[p['seed']] = p['prune70_drop'] * 100
    if 'lambda_max' in p:
        lam.setdefault(key, {})[p['seed']] = p['lambda_max']

# 기존 세션(rho sweep, seed 0)의 값 — 참고용
PREV = {('adamw', 0.0): 71.29, ('adamw', 0.40): 70.31,
        ('lion',  0.0): 68.70, ('lion',  0.40): 68.85,
        ('muon',  0.0): 11.33, ('muon',  0.40):  6.84}

print('=' * 78)
print('시드별 pr70 — 70% 가지치기 후 정확도 하락폭 (%p, 낮을수록 강건)')
print('=' * 78)
print(f"{'optimizer':8} {'rho':>5} | {'s0':>7} {'s1':>7} {'s2':>7} | "
      f"{'평균':>7} {'폭':>6} | {'이전세션':>8}")
print('-' * 78)

mean, spread = {}, {}
for opt in ['adamw', 'lion', 'muon']:
    for r in [0.0, 0.40]:
        d = pr70.get((opt, r), {})
        vals = [d.get(s) for s in (0, 1, 2)]
        got = [v for v in vals if v is not None]
        if len(got) < 2:
            continue
        mean[(opt, r)] = float(np.mean(got))
        spread[(opt, r)] = max(got) - min(got)
        cells_ = '  '.join(f'{v:6.2f}' if v is not None else '     -' for v in vals)
        print(f'{opt:8} {r:5g} | {cells_} | {np.mean(got):7.2f} '
              f'{max(got)-min(got):6.2f} | {PREV[(opt, r)]:8.2f}')

print()
print('=' * 78)
print('판정 — rho 효과가 시드 노이즈보다 큰가?')
print('=' * 78)
for opt in ['adamw', 'lion', 'muon']:
    if (opt, 0.0) not in mean or (opt, 0.40) not in mean:
        continue
    effect = mean[(opt, 0.0)] - mean[(opt, 0.40)]     # 양수 = rho가 개선시킴
    noise = max(spread[(opt, 0.0)], spread[(opt, 0.40)])
    ratio = effect / noise if noise > 0 else float('inf')
    verdict = ('OK    — 효과가 노이즈의 2배 이상. 주장 유지 가능' if ratio >= 2 else
               '애매  — 효과와 노이즈가 비슷. 시드를 더 늘려야' if ratio >= 1 else
               '위험  — 노이즈가 더 큼. 주장을 약하게 고쳐야')
    print(f'{opt:8} rho효과={effect:+6.2f}%p   시드폭={noise:5.2f}%p   '
          f'비율={ratio:5.2f}x   {verdict}')

print()
print('=' * 78)
print('부수 결과 — 골짜기(시드)마다 평탄도가 다른가?   [SGDR 아이디어 판단용]')
print('=' * 78)
for opt in ['adamw', 'lion', 'muon']:
    for r in [0.0, 0.40]:
        vals = sorted(lam.get((opt, r), {}).values())
        if len(vals) < 2:
            continue
        rel = (max(vals) - min(vals)) / np.mean(vals) * 100
        print(f'{opt:8} rho={r:<5g} lambda_max: ' +
              '  '.join(f'{v:7.2f}' for v in vals) +
              f'   (폭 {max(vals)-min(vals):6.2f}, 평균대비 {rel:.0f}%)')

print()
print('골짜기 간 폭이 크면 "여러 골짜기를 탐색해 고른다"는 아이디어에 근거가 생기고,')
print('작으면 어느 골짜기에 떨어지든 비슷하다는 뜻이라 고를 이유가 없다.')

시드별 pr70 — 70% 가지치기 후 정확도 하락폭 (%p, 낮을수록 강건)
optimizer   rho |      s0      s1      s2 |      평균      폭 |     이전세션
------------------------------------------------------------------------------
adamw        0 |  69.68   68.36   69.58 |   69.21   1.32 |    71.29
adamw      0.4 |  69.58   68.65   69.68 |   69.30   1.03 |    70.31
lion         0 |  68.36   69.78   68.75 |   68.96   1.42 |    68.70
lion       0.4 |  68.55   68.75   67.33 |   68.21   1.42 |    68.85
muon         0 |   9.57   11.28   11.77 |   10.87   2.20 |    11.33
muon       0.4 |   7.32    8.30   10.84 |    8.82   3.52 |     6.84

판정 — rho 효과가 시드 노이즈보다 큰가?
adamw    rho효과= -0.10%p   시드폭= 1.32%p   비율=-0.07x   위험  — 노이즈가 더 큼. 주장을 약하게 고쳐야
lion     rho효과= +0.75%p   시드폭= 1.42%p   비율= 0.53x   위험  — 노이즈가 더 큼. 주장을 약하게 고쳐야
muon     rho효과= +2.05%p   시드폭= 3.52%p   비율= 0.58x   위험  — 노이즈가 더 큼. 주장을 약하게 고쳐야

부수 결과 — 골짜기(시드)마다 평탄도가 다른가?   [SGDR 아이디어 판단용]
adamw    rho=0     lambda_max:   36.78    70.03    79.66   (폭  42.88, 평균대비 69%)
adamw

## 6. 결과 백업

`probes.json` 과 요약 CSV를 Output에 남긴다. 세션이 만료돼도 Output 탭에서 받을 수 있다.

In [7]:
import json, shutil, csv

probes = json.load(open(f'{RUNS}/probes.json'))
sv = {k: v for k, v in probes.items() if k.startswith('seedvar-')}

with open('/kaggle/working/seedvar_summary.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['run_id', 'optimizer', 'seed', 'rho', 'clean_acc',
                'lambda_max', 'sam_gap', 'basin_width',
                'int8_drop', 'int4_drop', 'prune50_drop', 'prune70_drop',
                'noise0.05_drop'])
    for k, p in sorted(sv.items()):
        r = 0.0 if p.get('sam_mode') == 'off' else p.get('rho', 0.0)
        w.writerow([k, p['optimizer'], p['seed'], r,
                    round(p.get('clean_acc', 0) * 100, 2),
                    round(p.get('lambda_max', 0), 4),
                    round(p.get('sam_gap', 0), 4),
                    round(p.get('basin_width', 0), 3),
                    round(p.get('int8_drop', 0) * 100, 2),
                    round(p.get('int4_drop', 0) * 100, 2),
                    round(p.get('prune50_drop', 0) * 100, 2),
                    round(p.get('prune70_drop', 0) * 100, 2),
                    round(p.get('noise0.05_drop', 0) * 100, 2)])

shutil.copy(f'{RUNS}/probes.json', '/kaggle/working/seedvar_probes.json')

print('Output에 저장됨:')
print('  seedvar_summary.csv   <- 이것만 받으면 충분')
print('  seedvar_probes.json')
print()
print(open('/kaggle/working/seedvar_summary.csv').read())

Output에 저장됨:
  seedvar_summary.csv   <- 이것만 받으면 충분
  seedvar_probes.json

run_id,optimizer,seed,rho,clean_acc,lambda_max,sam_gap,basin_width,int8_drop,int4_drop,prune50_drop,prune70_drop,noise0.05_drop
seedvar-cifar100-resnet18-adamw-sam_decay-s0-lr0.003_rho0.4,adamw,0,0.4,72.36,22.0932,0.7041,0.867,0.15,0.54,19.53,69.58,0.05
seedvar-cifar100-resnet18-adamw-sam_decay-s1-lr0.003_rho0.4,adamw,1,0.4,72.61,19.9506,0.749,0.867,0.15,0.34,15.53,68.65,0.24
seedvar-cifar100-resnet18-adamw-sam_decay-s2-lr0.003_rho0.4,adamw,2,0.4,73.58,18.3971,0.6041,0.867,0.0,1.9,16.7,69.68,0.49
seedvar-cifar100-resnet18-adamw-sam_off-s0-lr0.003,adamw,0,0.0,71.83,70.0291,0.0886,0.667,-0.15,0.98,21.68,69.68,0.2
seedvar-cifar100-resnet18-adamw-sam_off-s1-lr0.003,adamw,1,0.0,72.12,79.6629,0.0771,0.667,-0.15,1.66,23.34,68.36,-0.15
seedvar-cifar100-resnet18-adamw-sam_off-s2-lr0.003,adamw,2,0.0,71.63,36.7789,0.075,0.733,-0.24,0.83,23.39,69.58,-0.44
seedvar-cifar100-resnet18-lion-sam_decay-s0-lr0.0003_rho0.4,lion,0,0.4